# Pythia-410m: the regauge test

**GaugeMLA, our method, is the arm `mla_w64_ours_unrope_cost_kl`.** This notebook regauges Pythia-410m, builds MHA2MLA, TransMLA, CARE and GaugeMLA from each regauged checkpoint, and measures them at conversion: the paper's Tables 3 and 4 (see `experiments/regauge_test/README.md`).

Step 1 on a **High-RAM CPU** runtime; then switch to an **A100 GPU, High-RAM** runtime (Runtime → Change runtime type), rerun Setup, and run step 2. Writes to `MyDrive/gaugemla/regauge_test/p410m_rgkv/` (about 70 GB). Not yet run on Colab.

Requires a Hugging Face token: add it as the Colab secret `HF_TOKEN` and give this notebook access.

In [ ]:
# @title Setup
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
# The repository is public. While it is private, uncomment these lines and add the Colab secret READ_MLA_TOKEN
# (a GitHub token that can read it):
# from google.colab import userdata
# os.environ["READ_MLA_TOKEN"] = userdata.get("READ_MLA_TOKEN")
# with open("/content/askpass.sh", "w") as f:
#     f.write('#!/bin/sh\necho "$READ_MLA_TOKEN"\n')
# os.chmod("/content/askpass.sh", 0o700)
# os.environ["GIT_ASKPASS"] = "/content/askpass.sh"
os.environ["GIT_TERMINAL_PROMPT"] = "0"
if os.path.isdir("/content/repo/.git"):
    subprocess.run(["git", "-C", "/content/repo", "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", "/content/repo", "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                    "https://github.com/anon5159753/gaugemla.git", "/content/repo"], check=True)
    subprocess.run(["git", "-C", "/content/repo", "sparse-checkout", "set", "src", "colab"], check=True)
sys.path.insert(0, "/content/repo/colab")
import launch

In [ ]:
# @title 1. Build the initialisations, identity first (High-RAM CPU runtime)
for c in ("identity", "kvs", "kvd", "kvs_xf", "kvd_xf"):
    launch.run("p410m_rgkv", "init", DRY_RUN=False, REGAUGE=c)

In [ ]:
# @title 2. Score them (A100 GPU runtime: switch, rerun Setup, then this cell)
for c in ("identity", "kvs", "kvd", "kvs_xf", "kvd_xf"):
    launch.run("p410m_rgkv", "run", REGAUGE=c)